In [1]:
!pip install polars

In [8]:
import polars as pl

# Load source data if missing and prepare cleaned_df
if 'order_id_order_date_36' not in globals():
    order_id_order_date_36 = pl.read_csv('order_id_order_date_36.csv')

cleaned_df = order_id_order_date_36.with_columns([
    pl.col("customer_name").str.to_lowercase(),
    pl.col("channel").str.to_lowercase(),
    pl.col("status").str.to_lowercase()
]).unique()

# Extract time-series features using flexible parsing to handle mixed formats
cleaned_df = cleaned_df.with_columns([
    pl.col("order_date").str.to_datetime(strict=False).alias("order_datetime")
])

# If some dates failed (returned null), try a secondary parse for the DD-MM-YYYY format
cleaned_df = cleaned_df.with_columns(
    pl.when(pl.col("order_datetime").is_null())
    .then(pl.col("order_date").str.strptime(pl.Datetime, format="%d-%m-%Y", strict=False))
    .otherwise(pl.col("order_datetime"))
    .alias("order_datetime")
)

# Derive temporal features from the successfully parsed datetime
cleaned_df = cleaned_df.with_columns([
    pl.col("order_datetime").dt.year().alias("order_year"),
    pl.col("order_datetime").dt.month().alias("order_month"),
    pl.col("order_datetime").dt.day().alias("order_day"),
    pl.col("order_datetime").dt.weekday().alias("order_weekday"),
    pl.col("order_datetime").dt.week().alias("order_week"),
    pl.col("order_datetime").dt.quarter().alias("order_quarter")
])

display(cleaned_df.head())

order_id,order_date,customer_id,customer_name,email,product_id,product_title,category,quantity,price,total,channel,status,order_datetime,order_year,order_month,order_day,order_weekday,order_week,order_quarter
i64,str,str,str,str,i64,str,str,str,str,str,str,str,datetime[μs],i32,i8,i8,i8,i8,i8
10002,"""2024-01-03""","""2002""","""mark lee""","""mark23@gmail.com""",4001,"""Summer Dress""","""Fashion""","""2""","""45""","""90""","""facebook""","""paid""",2024-01-03 00:00:00,2024,1,3,3,1,1
10013,"""2024-01-09""","""2008""","""jane doe""","""jane.doe@bell.ca""",4006,"""T-Shirt""","""Fashion""","""1""","""19.99""","""19.99""","""organic""","""paid""",2024-01-09 00:00:00,2024,1,9,2,2,1
10006,"""2024-01-05""","""2004""","""david rossi""","""david@rogers.ca""",5003,"""Mascara""","""Beauty""","""1""","""NULL""","""19.99""","""organic""","""refunded""",2024-01-05 00:00:00,2024,1,5,5,1,1
10026,"""2024-01-15""","""2020""","""mason lewis""","""mason.l@gmail.com""",4002,"""summer dress""","""fashion""","""1""","""45""","""45""","""google""","""paid""",2024-01-15 00:00:00,2024,1,15,1,3,1
10032,"""2024-01-18""","""2026""","""michael wright""","""michael.w@yahoo.com""",5001,"""Lipstick""","""Beauty""","""3""","""14.99""","""44.97""","""organic""","""paid""",2024-01-18 00:00:00,2024,1,18,4,3,1
